# Mahurangi Flood-Risk Classification — K-Nearest Neighbours

KNN classifier for the 24-hour-ahead flood-risk target, using the same chronological split, training-only threshold, boundary-safe target and feature set as the Mahurangi Random Forest notebook, so the two models are directly comparable.

Input is Kabishcka's already-cleaned `mahurangi_cleaned_hourly.csv` (two rainfall gauges merged, missing discharge/water level recovered from a second station). Both river variables are used as features; water level is the target variable.

**What is different for KNN:** it is distance-based, so features must be scaled (scaler fitted on training data only); it has no class-weight option, so neighbours are distance-weighted; and `k` is tuned on the validation set.

## 1. Setup and Data Loading

In [1]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_auc_score,
    average_precision_score, precision_recall_curve, fbeta_score,
    precision_score, recall_score, accuracy_score,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

In [2]:
# Edit INPUT_PATH to match where the file is on your machine.
INPUT_PATH = "mahurangi_cleaned_hourly.csv"
TIME_COL = "timestamp"
RIVER_COLS = ['discharge_m3s', 'water_level_m']
TARGET_COL = "water_level_m"      # river variable used to define the flood-risk label
FORECAST_HORIZON = 24            # hours ahead the target looks
FLOOD_PERCENTILE = 0.995         # high-water percentile, from TRAINING data only
TRAIN_FRAC = 0.70
VAL_FRAC = 0.85                  # train ends at 0.70, validation at 0.85, rest is test
K_VALUES = [5, 15, 31, 61, 101, 151, 251, 401]  # candidate neighbour counts, tuned on validation

In [3]:
df = pd.read_csv(INPUT_PATH, parse_dates=[TIME_COL])
df = df.rename(columns={TIME_COL: "start_time"})
df = df.sort_values("start_time").reset_index(drop=True)

print("Rows:", len(df))
print("Start:", df["start_time"].min())
print("End:", df["start_time"].max())
df.head()

Rows: 96432
Start: 2015-01-01 01:00:00
End: 2026-01-01 00:00:00


,start_time,rainfall_mm,rainfall_substituted,rainfall_source,discharge_m3s,water_level_m,discharge_substituted,water_level_substituted,discharge_source,water_level_source
0,2015-01-01 01:00:00,0.000,0,644616_primary,0.323,1.332,0,0,6863_primary,6863_primary
1,2015-01-01 02:00:00,0.000,0,644616_primary,0.423,1.349,0,0,6863_primary,6863_primary
2,2015-01-01 03:00:00,1.250,0,644616_primary,0.461,1.355,0,0,6863_primary,6863_primary
3,2015-01-01 04:00:00,1.240,0,644616_primary,0.410,1.347,0,0,6863_primary,6863_primary
4,2015-01-01 05:00:00,0.000,0,644616_primary,0.374,1.341,0,0,6863_primary,6863_primary


## 2. Data Cleaning

The file is already merged and gap-recovered; the only remaining step is dropping rows where both river stations were missing at once.

In [4]:
rows_before = len(df)
df = df.dropna(subset=RIVER_COLS).reset_index(drop=True)
print("Rows dropped (river data missing):", rows_before - len(df))
print("Remaining rows:", len(df))

Rows dropped (river data missing): 90
Remaining rows: 96342


## 3. Chronological Split and Target Definition

No shuffling — later periods stay in later splits. The flood-risk threshold comes from the training period only, so no validation/test information leaks into the label.

In [5]:
n = len(df)
train_end = int(n * TRAIN_FRAC)
val_end = int(n * VAL_FRAC)

df["data_period"] = "test"
df.loc[df.index < train_end, "data_period"] = "train"
df.loc[(df.index >= train_end) & (df.index < val_end), "data_period"] = "validation"

high_water_threshold = df.loc[df["data_period"] == "train", TARGET_COL].quantile(FLOOD_PERCENTILE)
print(f"Training-period {FLOOD_PERCENTILE:.1%} threshold ({TARGET_COL}):", round(high_water_threshold, 3))

df.groupby("data_period").agg(rows=("start_time", "size"), beginning=("start_time", "min"),
                               ending=("start_time", "max")).loc[["train", "validation", "test"]]

Training-period 99.5% threshold (water_level_m): 2.424


,rows,beginning,ending
data_period,,,
train,67439,2015-01-01 01:00:00,2022-09-11 22:00:00
validation,14451,2022-09-11 23:00:00,2024-05-06 19:00:00
test,14452,2024-05-06 20:00:00,2026-01-01 00:00:00


### 24-hour-ahead target (boundary-safe)

`flood_next_24h` = 1 if the river variable exceeds the threshold at any point in the next 24 hours. Targets whose 24-hour window crosses into another split are invalidated.

In [6]:
future_values = pd.concat([df[TARGET_COL].shift(-h) for h in range(1, FORECAST_HORIZON + 1)], axis=1)
df["future_24h_max"] = future_values.max(axis=1, skipna=False)
df["flood_next_24h"] = np.where(df["future_24h_max"].isna(), np.nan,
                                (df["future_24h_max"] >= high_water_threshold).astype(int))

future_periods = pd.concat([df["data_period"].shift(-h) for h in range(1, FORECAST_HORIZON + 1)], axis=1)
same_period_target = future_periods.eq(df["data_period"], axis=0).all(axis=1)
df.loc[~same_period_target, ["future_24h_max", "flood_next_24h"]] = np.nan

target_balance = (df.dropna(subset=["flood_next_24h"]).groupby("data_period")["flood_next_24h"]
                  .agg(total_rows="count", flood_hours="sum", flood_percentage="mean"))
target_balance["flood_percentage"] *= 100
target_balance.loc[["train", "validation", "test"]]

,total_rows,flood_hours,flood_percentage
data_period,,,
train,67415,1298.000,1.925
validation,14427,816.000,5.656
test,14428,410.000,2.842


## 4. Feature Engineering

Rainfall lags and rolling sums, river-variable lags and rate-of-change, plus cyclical time-of-day / time-of-year features (same set as the Random Forest notebook).

In [7]:
LAG_HOURS = [1, 2, 3, 6, 12, 24]
RAIN_SUM_WINDOWS = [3, 6, 12, 24, 48]
CHANGE_PERIODS = [1, 3, 6]

for lag in LAG_HOURS:
    df[f"rainfall_lag_{lag}h"] = df["rainfall_mm"].shift(lag)
    for col in RIVER_COLS:
        df[f"{col}_lag_{lag}h"] = df[col].shift(lag)

for window in RAIN_SUM_WINDOWS:
    df[f"rainfall_sum_{window}h"] = df["rainfall_mm"].rolling(window=window, min_periods=window).sum()

for period in CHANGE_PERIODS:
    for col in RIVER_COLS:
        df[f"{col}_change_{period}h"] = df[col] - df[col].shift(period)

df["hour"] = df["start_time"].dt.hour
df["month"] = df["start_time"].dt.month
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["month_sin"] = np.sin(2 * np.pi * (df["month"] - 1) / 12)
df["month_cos"] = np.cos(2 * np.pi * (df["month"] - 1) / 12)

In [8]:
feature_columns = (
    ["rainfall_mm"] + RIVER_COLS
    + [f"rainfall_lag_{l}h" for l in LAG_HOURS]
    + [f"{c}_lag_{l}h" for c in RIVER_COLS for l in LAG_HOURS]
    + [f"rainfall_sum_{w}h" for w in RAIN_SUM_WINDOWS]
    + [f"{c}_change_{p}h" for c in RIVER_COLS for p in CHANGE_PERIODS]
    + ["hour_sin", "hour_cos", "month_sin", "month_cos"]
)

required_columns = feature_columns + ["flood_next_24h", "start_time", "data_period"]
modelling_dataset = df[required_columns].dropna().reset_index(drop=True)
modelling_dataset["flood_next_24h"] = modelling_dataset["flood_next_24h"].astype(int)

train_data = modelling_dataset[modelling_dataset["data_period"] == "train"]
validation_data = modelling_dataset[modelling_dataset["data_period"] == "validation"]
test_data = modelling_dataset[modelling_dataset["data_period"] == "test"]

X_train, y_train = train_data[feature_columns], train_data["flood_next_24h"]
X_validation, y_validation = validation_data[feature_columns], validation_data["flood_next_24h"]
X_test, y_test = test_data[feature_columns], test_data["flood_next_24h"]

print("Features:", len(feature_columns))
print("Training shape:", X_train.shape)
print("Validation shape:", X_validation.shape)
print("Test shape:", X_test.shape)

Features: 36
Training shape: (67368, 36)
Validation shape: (14427, 36)
Test shape: (14428, 36)


## 5. Scaling

KNN uses Euclidean distance, so a feature measured in large units would dominate. The scaler is **fitted on the training set only**, then applied to validation and test.

In [9]:
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_validation_s = scaler.transform(X_validation)
X_test_s = scaler.transform(X_test)

## 6. KNN: Tune k and the Decision Threshold on Validation

For each candidate `k` we fit on the training set, then find the probability threshold that maximises F2 on validation (F2 weights recall — catching real flood-risk periods — above precision). The `k` with the best validation F2 is selected. The test set is not used here.

In [10]:
def find_best_f2_threshold(y_true, probabilities):
    precision, recall, thresholds = precision_recall_curve(y_true, probabilities)
    precision, recall = precision[:-1], recall[:-1]
    f2_scores = 5 * precision * recall / (4 * precision + recall + 1e-12)
    return thresholds[np.argmax(f2_scores)]

rows = []
for k in K_VALUES:
    knn = KNeighborsClassifier(n_neighbors=k, weights="distance", n_jobs=-1)
    knn.fit(X_train_s, y_train)
    proba = knn.predict_proba(X_validation_s)[:, 1]
    thr = find_best_f2_threshold(y_validation, proba)
    pred = (proba >= thr).astype(int)
    rows.append({
        "k": k, "threshold": thr,
        "precision": precision_score(y_validation, pred, zero_division=0),
        "recall": recall_score(y_validation, pred, zero_division=0),
        "f2": fbeta_score(y_validation, pred, beta=2, zero_division=0),
        "roc_auc": roc_auc_score(y_validation, proba),
        "pr_auc": average_precision_score(y_validation, proba),
    })

k_results = pd.DataFrame(rows)
best_row = k_results.sort_values("f2", ascending=False).iloc[0]
best_k = int(best_row["k"])
best_threshold = float(best_row["threshold"])

print(f"Selected k = {best_k} (validation F2 = {best_row['f2']:.3f})")
k_results

Selected k = 401 (validation F2 = 0.457)


,k,threshold,precision,recall,f2,roc_auc,pr_auc
0,5,0.169,0.320,0.412,0.390,0.685,0.349
1,15,0.063,0.241,0.482,0.401,0.710,0.395
2,31,0.057,0.243,0.511,0.418,0.738,0.424
3,61,0.080,0.364,0.450,0.429,0.758,0.443
4,101,0.069,0.351,0.466,0.437,0.768,0.452
5,151,0.063,0.354,0.477,0.446,0.783,0.461
6,251,0.054,0.323,0.496,0.448,0.799,0.470
7,401,0.051,0.328,0.507,0.457,0.807,0.475


## 7. Final Test Evaluation

The test set is evaluated exactly once, using the `k` and threshold selected on validation.

In [11]:
final_knn = KNeighborsClassifier(n_neighbors=best_k, weights="distance", n_jobs=-1)
final_knn.fit(X_train_s, y_train)

test_probability = final_knn.predict_proba(X_test_s)[:, 1]
test_prediction = (test_probability >= best_threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, test_prediction).ravel()

test_results = pd.Series({
    "k": best_k,
    "selected_threshold": best_threshold,
    "accuracy": accuracy_score(y_test, test_prediction),
    "precision": precision_score(y_test, test_prediction, zero_division=0),
    "recall": recall_score(y_test, test_prediction, zero_division=0),
    "f2_score": fbeta_score(y_test, test_prediction, beta=2, zero_division=0),
    "roc_auc": roc_auc_score(y_test, test_probability),
    "pr_auc": average_precision_score(y_test, test_probability),
    "true_positives": tp, "false_negatives": fn,
    "false_positives": fp, "true_negatives": tn,
})

print("FINAL KNN — TEST RESULTS")
print("Test period was not used for training, k selection or threshold selection.\n")
test_results.round(4)

FINAL KNN — TEST RESULTS
Test period was not used for training, k selection or threshold selection.



k                      401.000
selected_threshold       0.051
accuracy                 0.922
precision                0.162
recall                   0.419
f2_score                 0.318
roc_auc                  0.775
pr_auc                   0.343
true_positives         172.000
false_negatives        238.000
false_positives        889.000
true_negatives       13129.000
dtype: float64

## 8. Export

In [12]:
from pathlib import Path

output_directory = Path("outputs/mahurangi")
output_directory.mkdir(parents=True, exist_ok=True)
test_results.to_frame("value").to_csv(output_directory / "knn_test_results.csv")
k_results.to_csv(output_directory / "knn_k_tuning.csv", index=False)
print("Saved KNN results to", output_directory)

Saved KNN results to outputs/mahurangi
